<a href="https://colab.research.google.com/github/matinapap/Multimedia-Systems/blob/main/Multimedia_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()  # διάλεξε το video.avi

Saving input_video.avi to input_video (1).avi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

video_path = '/content/drive/MyDrive/input_video.avi'

#!/usr/bin/env python3
import cv2
import numpy as np
import matplotlib.pyplot as plt
import argparse
import os
import sys
import heapq
from collections import Counter

# -------------------------------
# (A) Frame differences I_{n+1}=I_n
# -------------------------------
def compute_frame_differences(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    frame_diffs = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        diff = cv2.absdiff(frame_gray, prev_frame_gray)
        frame_diffs.append(diff)

        prev_frame_gray = frame_gray

    cap.release()
    return frame_diffs

def visualize_frame_differences(frame_diffs, output_dir, prefix="diff"):
    os.makedirs(output_dir, exist_ok=True)
    for i, diff in enumerate(frame_diffs):
        plt.imshow(diff, cmap='gray')
        plt.axis('off')
        plt.title(f'{prefix} {i}')
        plt.savefig(os.path.join(output_dir, f'{prefix}_{i:04d}.png'))
        plt.close()

# -------------------------------
# (B) Huffman Coding
# -------------------------------
def compute_huffman(frame_diffs, output_dir="huffman_output"):
    os.makedirs(output_dir, exist_ok=True)

    freq = Counter()
    total_symbols = 0
    for diff in frame_diffs:
        arr = diff.ravel()
        freq.update(arr.tolist())
        total_symbols += arr.size

    if total_symbols == 0:
        print("No symbols found in frame_diffs.")
        return {}, 0, 0

    heap = []
    uid = 0
    for sym, f in freq.items():
        heap.append((f, uid, sym))
        uid += 1
    heapq.heapify(heap)

    if len(heap) == 1:
        _, _, only_sym = heap[0]
        codes = {int(only_sym): "0"}
    else:
        while len(heap) > 1:
            f1, id1, node1 = heapq.heappop(heap)
            f2, id2, node2 = heapq.heappop(heap)
            combined = (f1 + f2, uid, (node1, node2))
            uid += 1
            heapq.heappush(heap, combined)
        _, _, root = heap[0]

        codes = {}
        def assign_codes(node, prefix):
            if isinstance(node, int) or isinstance(node, np.integer):
                codes[int(node)] = prefix if prefix != "" else "0"
                return
            left, right = node
            assign_codes(left, prefix + "0")
            assign_codes(right, prefix + "1")
        assign_codes(root, "")

    compressed_bits = sum(freq[sym] * len(codes[int(sym)]) for sym in freq)
    original_bits = total_symbols * 8

    codebook_path = os.path.join(output_dir, "huffman_codebook.txt")
    with open(codebook_path, "w") as fh:
        fh.write("symbol\tfrequency\tcode\tcode_length\n")
        for sym, f in freq.most_common():
            fh.write(f"{int(sym)}\t{f}\t{codes[int(sym)]}\t{len(codes[int(sym)])}\n")

    print(f"Huffman codebook saved to: {codebook_path}")
    print(f"Original bits: {original_bits}")
    print(f"Compressed bits: {compressed_bits}")
    if compressed_bits > 0:
        print(f"Compression ratio: {original_bits / compressed_bits:.4f}")
        print(f"Average bits/symbol: {compressed_bits / total_symbols:.4f}")

    return codes, original_bits, compressed_bits

# -------------------------------
# (C) Motion Compensation
# -------------------------------
# def block_matching(prev_frame, curr_frame, block_size=16, search_range=8):
#     """Brute-force block matching with SAD"""
#     h, w = curr_frame.shape
#     pred_frame = np.zeros_like(curr_frame)

#     for y in range(0, h, block_size):
#         for x in range(0, w, block_size):
#             # skip incomplete blocks at borders
#             if y + block_size > h or x + block_size > w:
#                 continue

#             block = curr_frame[y:y+block_size, x:x+block_size]
#             best_sad = float("inf")
#             best_match = (y, x)

#             for dy in range(-search_range, search_range+1):
#                 for dx in range(-search_range, search_range+1):
#                     ref_y = y + dy
#                     ref_x = x + dx
#                     if (ref_y < 0 or ref_x < 0 or
#                         ref_y + block_size > h or
#                         ref_x + block_size > w):
#                         continue
#                     ref_block = prev_frame[ref_y:ref_y+block_size,
#                                            ref_x:ref_x+block_size]
#                     sad = np.sum(np.abs(block.astype(int) - ref_block.astype(int)))
#                     if sad < best_sad:
#                         best_sad = sad
#                         best_match = (ref_y, ref_x)

#             ref_y, ref_x = best_match
#             pred_frame[y:y+block_size, x:x+block_size] = prev_frame[ref_y:ref_y+block_size,
#                                                                     ref_x:ref_x+block_size]
#     return pred_frame

# def compute_motion_compensation(video_path, block_size=16, search_range=8):
#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         raise IOError(f"Cannot open video: {video_path}")

#     ret, prev_frame = cap.read()
#     if not ret:
#         raise IOError("Cannot read the first frame from the video")

#     prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
#     mc_diffs = []

#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break
#         curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

#         pred = block_matching(prev_gray, curr_gray,
#                               block_size=block_size,
#                               search_range=search_range)
#         diff = cv2.absdiff(curr_gray, pred)
#         mc_diffs.append(diff)

#         prev_gray = curr_gray

#     cap.release()
#     return mc_diffs

# -------------------------------
# MAIN
# -------------------------------
def main():
    parser = argparse.ArgumentParser(description="Video coding project (A,B,C)")
    parser.add_argument("video_path", help="Path to input .avi video")
    parser.add_argument("--output_dir", default="results", help="Output directory")
    parser.add_argument("--block_size", type=int, default=16, help="Block size for motion compensation")
    parser.add_argument("--search_range", type=int, default=8, help="Search range for motion compensation")
    args, unknown = parser.parse_known_args()


    # (A) Frame differences
    diffs_dir = os.path.join(args.output_dir, "frame_diffs")
    print("Computing frame differences...")
    frame_diffs = compute_frame_differences(args.video_path)
    print(f"Computed {len(frame_diffs)} frame differences.")
    visualize_frame_differences(frame_diffs, diffs_dir, prefix="diff")

    # (B) Huffman coding (για τα frame_diffs του Α)
    compute_huffman(frame_diffs, os.path.join(args.output_dir, "huffman_A"))

    # # (C) Motion-compensated differences
    # mc_dir = os.path.join(args.output_dir, "mc_diffs")
    # print("Computing motion-compensated frame differences...")
    # mc_diffs = compute_motion_compensation(args.video_path,
    #                                        block_size=args.block_size,
    #                                        search_range=args.search_range)
    # print(f"Computed {len(mc_diffs)} MC frame differences.")
    # visualize_frame_differences(mc_diffs, mc_dir, prefix="mc_diff")
    # compute_huffman(mc_diffs, os.path.join(args.output_dir, "huffman_C"))

if __name__ == "__main__":
    if len(sys.argv) == 1:
        sys.argv = ["script.py", video_path, "--output_dir", "results"]
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing frame differences...


OSError: Cannot open video: /root/.local/share/jupyter/runtime/kernel-a76da3cb-14db-4a99-b739-bae88aabd0f8.json

In [ ]:
# -------------------------------
# Google Drive mount & έλεγχος αρχείων
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

import os

video_path = '/content/drive/MyDrive/input_video.avi'
print("Αρχεία στον φάκελο MyDrive:")
!ls /content/drive/MyDrive/

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Δεν βρέθηκε το βίντεο στο path: {video_path}")

# -------------------------------
# Import βιβλιοθήκες
# -------------------------------
import cv2
import numpy as np
import matplotlib.pyplot as plt
import heapq
from collections import Counter

# -------------------------------
# (A) Frame differences
# -------------------------------
def compute_frame_differences(video_path):
    # φορτώνει το βίντεο
    cap = cv2.VideoCapture(video_path)

    # ελέγχει αν το βίντεο ανοίγει
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    # μετατρέπει το πρώτο frame σε grayscale
    prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    # λίστα που αποθηκεύει όλα τα frame differences
    frame_diffs = []

    while True:
        #διαβάζει το επόμενο frame
        ret, frame = cap.read()
        if not ret:
            break
        #μετατρέπει το τωρινό frame σε grayscale
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(frame_gray, prev_frame_gray)
        frame_diffs.append(diff)
        prev_frame_gray = frame_gray

    cap.release()
    return frame_diffs

def visualize_frame_differences(frame_diffs, output_dir, prefix="diff"):
    os.makedirs(output_dir, exist_ok=True)
    for i, diff in enumerate(frame_diffs):
        plt.imshow(diff, cmap='gray')
        plt.axis('off')
        plt.title(f'{prefix} {i}')
        plt.savefig(os.path.join(output_dir, f'{prefix}_{i:04d}.png'))
        plt.close()

# -------------------------------
# (B) Huffman coding
# -------------------------------
def compute_huffman(frame_diffs, output_dir="huffman_output"):
    os.makedirs(output_dir, exist_ok=True)

    freq = Counter()
    total_symbols = 0
    for diff in frame_diffs:
        arr = diff.ravel()
        freq.update(arr.tolist())
        total_symbols += arr.size

    if total_symbols == 0:
        print("No symbols found in frame_diffs.")
        return {}, 0, 0

    heap = []
    uid = 0
    for sym, f in freq.items():
        heap.append((f, uid, sym))
        uid += 1
    heapq.heapify(heap)

    if len(heap) == 1:
        _, _, only_sym = heap[0]
        codes = {int(only_sym): "0"}
    else:
        while len(heap) > 1:
            f1, id1, node1 = heapq.heappop(heap)
            f2, id2, node2 = heapq.heappop(heap)
            combined = (f1 + f2, uid, (node1, node2))
            uid += 1
            heapq.heappush(heap, combined)
        _, _, root = heap[0]

        codes = {}
        def assign_codes(node, prefix):
            if isinstance(node, int) or isinstance(node, np.integer):
                codes[int(node)] = prefix if prefix != "" else "0"
                return
            left, right = node
            assign_codes(left, prefix + "0")
            assign_codes(right, prefix + "1")
        assign_codes(root, "")

    compressed_bits = sum(freq[sym] * len(codes[int(sym)]) for sym in freq)
    original_bits = total_symbols * 8

    codebook_path = os.path.join(output_dir, "huffman_codebook.txt")
    with open(codebook_path, "w") as fh:
        fh.write("symbol\tfrequency\tcode\tcode_length\n")
        for sym, f in freq.most_common():
            fh.write(f"{int(sym)}\t{f}\t{codes[int(sym)]}\t{len(codes[int(sym)])}\n")

    print(f"Huffman codebook saved to: {codebook_path}")
    print(f"Original bits: {original_bits}")
    print(f"Compressed bits: {compressed_bits}")
    if compressed_bits > 0:
        print(f"Compression ratio: {original_bits / compressed_bits:.4f}")
        print(f"Average bits/symbol: {compressed_bits / total_symbols:.4f}")

    return codes, original_bits, compressed_bits

# -------------------------------
# Εκτέλεση για Colab
# -------------------------------
output_dir = "results"

# (A) Frame differences
print("Computing frame differences...")
frame_diffs = compute_frame_differences(video_path)
print(f"Computed {len(frame_diffs)} frame differences.")

diffs_dir = os.path.join(output_dir, "frame_diffs")
visualize_frame_differences(frame_diffs, diffs_dir, prefix="diff")

# (B) Huffman coding
huffman_dir = os.path.join(output_dir, "huffman_A")
compute_huffman(frame_diffs, huffman_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Αρχεία στον φάκελο MyDrive:
'Colab Notebooks'     'ΔΙΔΑΚΤΙΚΗ ΤΗΣ ΠΛΗΡΟΦΟΡΙΚΗΣ'
 input_video.avi      'Ευφυείς Πράκτορες'
 Podcast.odt	      'Υπολογιστικό φύλλο χωρίς τίτλο (1).gsheet'
 speech_and_audio     'Υπολογιστικό φύλλο χωρίς τίτλο.gsheet'
 UCD_RecruitmentTool
Computing frame differences...
Computed 113 frame differences.
Huffman codebook saved to: results/huffman_A/huffman_codebook.txt
Original bits: 1874534400
Compressed bits: 690429588
Compression ratio: 2.7150
Average bits/symbol: 2.9466


({3: '0000',
  9: '000100',
  15: '0001010',
  45: '0001011000',
  46: '0001011001',
  32: '000101101',
  22: '00010111',
  5: '00011',
  31: '001000000',
  64: '00100000100',
  145: '0010000010100000',
  156: '00100000101000010',
  157: '00100000101000011',
  137: '001000001010001',
  123: '00100000101001',
  125: '00100000101010',
  126: '00100000101011',
  78: '001000001011',
  60: '00100000110',
  59: '00100000111',
  102: '0010000100000',
  108: '0010000100001',
  77: '001000010001',
  61: '00100001001',
  44: '0010000101',
  62: '00100001100',
  103: '0010000110100',
  107: '0010000110101',
  76: '001000011011',
  63: '00100001110',
  73: '001000011110',
  165: '001000011111000000',
  174: '0010000111110000010',
  175: '0010000111110000011',
  167: '001000011111000010',
  173: '0010000111110000110',
  176: '0010000111110000111',
  144: '0010000111110001',
  155: '00100001111100100',
  152: '00100001111100101',
  153: '00100001111100110',
  154: '00100001111100111',
  127: '001000